In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, mean_squared_error, f1_score
import numpy as np
import warnings
import joblib
import os

warnings.filterwarnings('ignore')

d:\Material\Programming\Result\DEPLOYED\avalon-portfolio\stefano-budi-portfolio\projects\fed_speech\fed-speech_VENV\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Create Merged DataFrame

In [11]:
embeddings = np.load("fed_speech_embeddings.npy")

fed_speech = pd.read_csv("dataset/fed_speech_parsed_1966.csv")
dates = pd.to_datetime(fed_speech["date"])

prices = pd.read_csv("dataset/price_action.csv")
prices["date"] = pd.to_datetime(prices["date"])

In [12]:
fed_speech['embedding'] = list(embeddings)
fed_speech['date'] = pd.to_datetime(fed_speech['date'])

In [13]:
macro = pd.read_csv('dataset/macro_indicators.csv')

macro['date'] = pd.to_datetime(macro['date'])

macro = macro.set_index('date').sort_index()
macro["unemployment"] = macro["unemployment"].shift(30)
macro["growth_rate"]  = macro["growth_rate"].shift(30)

daily_index = pd.date_range(start=macro.index.min(), end=macro.index.max(), freq='D')

macro_daily = macro.reindex(daily_index).ffill().reset_index()
macro_daily = macro_daily.rename(columns={'index': 'date'})

In [14]:
merged_df = pd.merge(prices, macro_daily, on="date")

In [15]:
target_front = ['date', 'unemployment', 'interest_rate', 'growth_rate',
                'SPX','TNX','GOLD','VIX','DXY'] 
all_cols = merged_df.columns.tolist()
remaining_cols = [c for c in all_cols if c not in target_front]

merged_df = merged_df[target_front + remaining_cols]

In [16]:
for col in ['SPX','GOLD','TNX','DXY','VIX']:
    merged_df[f'{col}_ret'] = np.log(
        merged_df[col] / merged_df[col].shift(1)
    )

for col in ['SPX', 'GOLD', 'TNX', 'DXY', 'VIX']:
    merged_df[f'{col}_mom_3'] = (
        merged_df[col].shift(1) / merged_df[col].shift(4) - 1
    )
    merged_df[f'{col}_mom_7'] = (
        merged_df[col].shift(1) / merged_df[col].shift(8) - 1
    )
    merged_df[f'{col}_mom_30'] = (
        merged_df[col].shift(1) / merged_df[col].shift(31) - 1
    )

    merged_df[f'{col}_t-3'] = (
        merged_df[f'{col}_ret']
        .shift(1)
        .rolling(3)
        .mean()
    )
    merged_df[f'{col}_t-7'] = (
        merged_df[f'{col}_ret']
        .shift(1)
        .rolling(7)
        .mean()
    )
    merged_df[f'{col}_t-30'] = (
        merged_df[f'{col}_ret']
        .shift(1)
        .rolling(30)
        .mean()
    )

    merged_df[f'{col}_vol_7'] = (
        merged_df[f'{col}_ret'].shift(1).rolling(7).std()
    )
    merged_df[f'{col}_vol_30'] = (
        merged_df[f'{col}_ret'].shift(1).rolling(30).std()
    )

for col in ['SPX','GOLD','TNX','DXY','VIX']:
    merged_df[f'{col}_t+3'] = (
        merged_df[col].shift(-4) /
        merged_df[col].shift(-1)
    ) - 1

    merged_df[f'{col}_t+7'] = (
        merged_df[col].shift(-8) /
        merged_df[col].shift(-1)
    ) - 1

    merged_df[f'{col}_t+30'] = (
        merged_df[col].shift(-31) /
        merged_df[col].shift(-1)
    ) - 1
    
lag_cols = [
    col for col in fed_speech.columns
    if "_t-" in col or
       "unemployment" in col or
       "fed interest rate" in col or
       "growth rate" in col
]

fed_speech[lag_cols] = fed_speech[lag_cols].shift(1)

merged_df = merged_df.drop(columns=[
    'SPX_ret','GOLD_ret','TNX_ret','DXY_ret','VIX_ret',
    'SPX','GOLD','TNX','DXY','VIX'
])

In [17]:
fed_speech = fed_speech.merge(merged_df, on='date', how='left')

In [18]:
fed_speech = fed_speech.drop(columns=['content'])

In [19]:
emb_matrix = np.vstack(fed_speech["embedding"].values)

emb_df = pd.DataFrame(
    emb_matrix,
    index=fed_speech.index,
    columns=[f"emb_{i}" for i in range(emb_matrix.shape[1])]
)

fed_speech = pd.concat(
    [fed_speech.drop(columns=["embedding"]), emb_df],
    axis=1
)

# 2. Modeling

## A. Train-Test Split Preparation

In [20]:
fed_speech = fed_speech.sort_values("date").reset_index(drop=True)
fed_speech["date"] = pd.to_datetime(fed_speech["date"])

In [21]:
def get_weight(speaker):
    speaker = speaker.lower()
    
    if "chair" in speaker and "vice" not in speaker:
        return 3
    elif "vice chair" in speaker or "vice chairman" in speaker:
        return 2
    elif "governor" in speaker:
        return 1
    else:
        return 0.5

fed_speech["sample_weight"] = fed_speech["speaker"].apply(get_weight)

In [22]:
split_date = "2023-01-01"

train_df = fed_speech[fed_speech["date"] < split_date]
test_df  = fed_speech[(fed_speech["date"] >= split_date) & (fed_speech["date"] <= '2025-12-31')]

In [23]:
target_cols = [
    "SPX_t+3", "SPX_t+7", "SPX_t+30",
    "GOLD_t+3","GOLD_t+7","GOLD_t+30",
    "VIX_t+3","VIX_t+7","VIX_t+30",
    "TNX_t+3","TNX_t+7","TNX_t+30",
]

drop_cols = ["date", "title", "speaker", "sample_weight", "id"] + \
            target_cols + \
            ["DXY_t+3","DXY_t+7","DXY_t+30"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df[target_cols]
w_train = train_df["sample_weight"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df[target_cols]

## B. Modeling

### With Optuna (Used because better)

In [24]:
def objective(trial, X, y, sample_weights, asset, horizon):
    
    # Asset-aware search space (KEY CHANGE from last time)
    if asset in ["SPX", "VIX"]:
        max_depth        = trial.suggest_int('max_depth', 2, 4)
        max_leaf_nodes   = trial.suggest_int('max_leaf_nodes', 12, 25)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 30, 80)
        l2               = trial.suggest_float('l2_regularization', 2.0, 8.0, log=True)
        lr               = trial.suggest_float('learning_rate', 0.005, 0.02, log=True)
    elif asset == "GOLD":
        max_depth        = trial.suggest_int('max_depth', 2, 3)
        max_leaf_nodes   = trial.suggest_int('max_leaf_nodes', 10, 18)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 50, 100)
        l2               = trial.suggest_float('l2_regularization', 4.0, 10.0, log=True)
        lr               = trial.suggest_float('learning_rate', 0.005, 0.015, log=True)
    else:  # TNX, DXY
        max_depth        = 2
        max_leaf_nodes   = 10
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 60, 120)
        l2               = trial.suggest_float('l2_regularization', 5.0, 10.0, log=True)
        lr               = 0.01

    params = {
        'max_iter':          trial.suggest_int('max_iter', 500, 1500, step=100),
        'learning_rate':     lr,
        'max_depth':         max_depth,
        'max_leaf_nodes':    max_leaf_nodes,
        'min_samples_leaf':  min_samples_leaf,
        'l2_regularization': l2,
        'max_bins':          255,
        'early_stopping':    True,
        'n_iter_no_change':  50,
        'validation_fraction': 0.15,
        'random_state':      42
    }

    tscv = TimeSeriesSplit(n_splits=5)
    cv_aucs = []

    for train_idx, val_idx in tscv.split(X):
        purged_train_idx = train_idx[:-horizon] if horizon > 0 else train_idx
        if len(purged_train_idx) < 50:
            continue

        X_fold_train = X.iloc[purged_train_idx]
        y_fold_train = y.iloc[purged_train_idx]
        w_fold_train = sample_weights.iloc[purged_train_idx]
        X_fold_val, y_fold_val = X.iloc[val_idx], y.iloc[val_idx]

        model = HistGradientBoostingRegressor(**params)
        model.fit(X_fold_train, y_fold_train, sample_weight=w_fold_train)
        y_pred = model.predict(X_fold_val)

        y_true_binary = (y_fold_val > 0).astype(int)
        if len(np.unique(y_true_binary)) == 2:
            cv_aucs.append(roc_auc_score(y_true_binary, y_pred))

    trial.set_user_attr('auc', np.mean(cv_aucs) if cv_aucs else 0.0)
    return np.mean(cv_aucs) if cv_aucs else 0.0

In [ ]:
models = {}
best_params = {}
cv_metrics = {}

for col in target_cols:
    asset = col.split("_")[0]
    horizon = int(col.split("+")[1])

    print(f"\n{'='*70}")
    print(f"Optimizing: {col}  (asset: {asset}, horizon: {horizon})")
    print(f"{'='*70}")

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    study.optimize(
        lambda trial: objective(trial, X_train, y_train[col], w_train, asset, horizon),
        n_trials=20,
        show_progress_bar=True
    )

    best_trial = study.best_trial
    best_auc   = best_trial.user_attrs['auc']
    print(f"\nBest CV AUC: {best_auc:.4f}")

    best_params[col] = study.best_params
    cv_metrics[col]  = {'auc': best_auc}

    final_params = {
        **study.best_params,
        'max_bins':            255,
        'early_stopping':      True,
        'n_iter_no_change':    50,
        'validation_fraction': 0.15,
        'random_state':        42
    }

    model = HistGradientBoostingRegressor(**final_params)
    model.fit(X_train, y_train[col], sample_weight=w_train)
    models[col] = model

    print(f"✓ Complete!")

### Without Optuna

In [26]:
# models = {}

# for col in target_cols:
#     m = HistGradientBoostingRegressor(
#         max_iter=1500,          # allow slow learning
#         learning_rate=0.01,     # VERY important for noisy signal
#         max_depth=3,            # shallow = generalization
#         max_leaf_nodes=15,      # prevent signal memorization
        
#         min_samples_leaf=80,    # suppress macro noise fitting
#         l2_regularization=5.0,  # penalize overreaction
        
#         max_bins=255,           # capture subtle regime shifts
#         interaction_cst=None,   # allow cross-asset interactions
        
#         early_stopping=True,
#         n_iter_no_change=50,
#         validation_fraction=0.15,

#         random_state=42
#     )

#     m.fit(X_train, y_train[col], sample_weight=w_train)
#     models[col] = m

# 3. Evaluation

In [27]:
# from sklearn.metrics import mean_squared_error
# import numpy as np

# y_pred = np.column_stack([
#     models[col].predict(X_test)
#     for col in target_cols
# ])

# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# print("RMSE:", rmse)

# direction_acc = (np.sign(y_pred) == np.sign(y_test)).mean().mean()
# print("Directional Accuracy:", direction_acc)

In [28]:
# y_pred = pd.DataFrame(
#     y_pred,
#     columns=y_test.columns,
#     index=y_test.index
# )

# for col in target_cols:
#     rmse_col = np.sqrt(
#         mean_squared_error(
#             y_test[col],
#             y_pred[col]
#         )
#     )
    
#     acc = (
#         np.sign(y_pred[col]) ==
#         np.sign(y_test[col])
#     ).mean()
    
#     print(f"SIGN - {col}, {acc}")
#     print(f"RMSE - {col}, {rmse_col}")

In [29]:
# from sklearn.metrics import roc_auc_score, mean_squared_error
# import numpy as np

# assets = {
#     "SPX": ["SPX_t+3", "SPX_t+7", "SPX_t+30"],
#     "GOLD": ["GOLD_t+3", "GOLD_t+7", "GOLD_t+30"],
#     "VIX": ["VIX_t+3", "VIX_t+7", "VIX_t+30"],
#     "TNX": ["TNX_t+3", "TNX_t+7", "TNX_t+30"]
# }

# roles = {
#     "Governors":   lambda df: df[df["speaker"].str.lower().str.contains("governor", na=False)],
#     "Vice Chairs": lambda df: df[df["speaker"].str.lower().str.contains("vice chair", na=False)],
#     "Chairs":      lambda df: df[df["speaker"].str.lower().str.contains("chair", na=False)],
# }

# def get_metrics(true_series, pred_series):
#     auc = roc_auc_score((true_series > 0).astype(int), pred_series)
#     acc = (np.sign(pred_series) == np.sign(true_series)).mean()
#     rmse = np.sqrt(mean_squared_error(true_series, pred_series))
#     return auc, acc, rmse

# for asset, horizons in assets.items():
#     # Assign all horizons first
#     for col in horizons:
#         test_df[f"{col}_true"] = y_test[col]
#         test_df[f"{col}_pred"] = y_pred[col]

#     # Overall across ALL horizons for this asset
#     all_true = pd.concat([test_df[f"{col}_true"] for col in horizons])
#     all_pred = pd.concat([test_df[f"{col}_pred"] for col in horizons])
#     asset_auc, asset_acc, asset_rmse = get_metrics(all_true, all_pred)

#     print(f"\n{'='*95}")
#     print(f"{asset}  |  Overall → AUC: {asset_auc:.3f}  |  Acc: {asset_acc:.3f}  |  RMSE: {asset_rmse:.4f}")
#     print(f"{'='*95}")

#     # ── ROC-AUC table ──────────────────────────────────────────────────────
#     print(f"\n  ROC-AUC")
#     print(f"  {'Target':<15} {'Overall':<10} {'Governors':<12} {'Vice Chairs':<14} {'Chairs'}")
#     print(f"  {'-'*65}")
#     for col in horizons:
#         overall_auc, _, _ = get_metrics(test_df[f"{col}_true"], test_df[f"{col}_pred"])
#         role_aucs = {
#             name: get_metrics(role_filter(test_df)[f"{col}_true"],
#                               role_filter(test_df)[f"{col}_pred"])[0]
#             for name, role_filter in roles.items()
#         }
#         print(f"  {col:<15} {overall_auc:<10.3f} {role_aucs['Governors']:<12.3f} "
#               f"{role_aucs['Vice Chairs']:<14.3f} {role_aucs['Chairs']:.3f}")

#     # ── Directional Accuracy table ─────────────────────────────────────────
#     print(f"\n  Directional Accuracy")
#     print(f"  {'Target':<15} {'Overall':<10} {'Governors':<12} {'Vice Chairs':<14} {'Chairs'}")
#     print(f"  {'-'*65}")
#     for col in horizons:
#         _, overall_acc, _ = get_metrics(test_df[f"{col}_true"], test_df[f"{col}_pred"])
#         role_accs = {
#             name: get_metrics(role_filter(test_df)[f"{col}_true"],
#                               role_filter(test_df)[f"{col}_pred"])[1]
#             for name, role_filter in roles.items()
#         }
#         print(f"  {col:<15} {overall_acc:<10.3f} {role_accs['Governors']:<12.3f} "
#               f"{role_accs['Vice Chairs']:<14.3f} {role_accs['Chairs']:.3f}")

#     # ── RMSE table ─────────────────────────────────────────────────────────
#     print(f"\n  RMSE")
#     print(f"  {'Target':<15} {'Overall':<10} {'Governors':<12} {'Vice Chairs':<14} {'Chairs'}")
#     print(f"  {'-'*65}")
#     for col in horizons:
#         _, _, overall_rmse = get_metrics(test_df[f"{col}_true"], test_df[f"{col}_pred"])
#         role_rmses = {
#             name: get_metrics(role_filter(test_df)[f"{col}_true"],
#                               role_filter(test_df)[f"{col}_pred"])[2]
#             for name, role_filter in roles.items()
#         }
#         print(f"  {col:<15} {overall_rmse:<10.4f} {role_rmses['Governors']:<12.4f} "
#               f"{role_rmses['Vice Chairs']:<14.4f} {role_rmses['Chairs']:.4f}")

# 4. Parallel-Stage Prediction

### i. Imbalance Check

In [30]:
for col in target_cols:
    y_bin = (y_train[col].dropna() > 0).astype(int)
    print(f"{col:<12} up={y_bin.mean():.3f}  down={1-y_bin.mean():.3f}  n={len(y_bin)}")

SPX_t+3      up=0.567  down=0.433  n=1622
SPX_t+7      up=0.572  down=0.428  n=1622
SPX_t+30     up=0.644  down=0.356  n=1622
GOLD_t+3     up=0.506  down=0.494  n=1622
GOLD_t+7     up=0.518  down=0.482  n=1622
GOLD_t+30    up=0.522  down=0.478  n=1622
VIX_t+3      up=0.446  down=0.554  n=1622
VIX_t+7      up=0.430  down=0.570  n=1622
VIX_t+30     up=0.420  down=0.580  n=1622
TNX_t+3      up=0.486  down=0.514  n=1622
TNX_t+7      up=0.514  down=0.486  n=1622
TNX_t+30     up=0.488  down=0.512  n=1622


### ii. Execution

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score

def get_class_weight_multiplier(y_bin, neg_boost=1.0):
    """
    Per-row multiplier that upweights the minority (down) class via
    inverse class frequency. neg_boost lets you push further beyond
    pure balancing if F1 is still skewed toward the majority class.
    neg_boost=1.0 is plain inverse-frequency balancing;
    >1.0 pushes extra weight onto down days beyond just correcting for count.
    """
    n_pos = y_bin.sum()
    n_neg = len(y_bin) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.ones(len(y_bin))

    w_pos = len(y_bin) / (2.0 * n_pos)
    w_neg = len(y_bin) / (2.0 * n_neg) * neg_boost
    return np.where(y_bin == 1, w_pos, w_neg)


def objective_clf_purged(trial, X, y_bin, sample_weights, horizon):
    params = {
        'max_iter':          trial.suggest_int('max_iter', 200, 800, step=50),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 2, 5),
        'max_leaf_nodes':    trial.suggest_int('max_leaf_nodes', 8, 31),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 20, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10.0, log=True),
        'max_bins': 255,
        'early_stopping': True,
        'n_iter_no_change': 30,
        'validation_fraction': 0.15,
        'random_state': 42,
    }

    tscv = TimeSeriesSplit(n_splits=5)
    fold_f1s = []

    for tr_idx, val_idx in tscv.split(X):
        purged_tr_idx = tr_idx[:-horizon] if horizon > 0 else tr_idx
        if len(purged_tr_idx) < 50:
            continue

        y_tr, y_val = y_bin[purged_tr_idx], y_bin[val_idx]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_val)) < 2:
            continue

        w_tr = sample_weights[purged_tr_idx]

        model = HistGradientBoostingClassifier(**params)
        model.fit(X.iloc[purged_tr_idx], y_tr, sample_weight=w_tr)
        proba = model.predict_proba(X.iloc[val_idx])[:, 1]

        # sweep thresholds on this fold, keep the best macro-F1 —
        # 0.5 isn't guaranteed optimal once classes are weighted
        best_f1 = 0.0
        for t in np.linspace(0.1, 0.9, 17):
            pred = (proba >= t).astype(int)
            best_f1 = max(best_f1, f1_score(y_val, pred, average="macro", zero_division=0))

        fold_f1s.append(best_f1)

    return np.mean(fold_f1s) if fold_f1s else 0.0


clf_models = {}
clf_thresholds = {}
clf_best_params = {}

NEG_BOOST = 1.0

for col in target_cols:
    horizon = int(col.split("+")[1])
    y_col = y_train[col]
    valid = y_col.notna()

    X_tr_full = X_train[valid]
    y_tr_full = y_col[valid]
    y_bin_full = (y_tr_full > 0).astype(int).values

    class_w = get_class_weight_multiplier(y_bin_full, neg_boost=NEG_BOOST)
    w_clf_full = w_train[valid].values * class_w   # speaker importance × class-imbalance

    print(f"\n{'='*60}\nTuning classifier: {col} (horizon={horizon})\n{'='*60}")
    print(f"  class balance: {y_bin_full.mean():.3f} up / {1 - y_bin_full.mean():.3f} down")

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        lambda trial: objective_clf_purged(trial, X_tr_full, y_bin_full, w_clf_full, horizon),
        n_trials=20, show_progress_bar=False
    )

    best_params = study.best_params
    clf_best_params[col] = best_params
    print(f"  Best CV macro-F1: {study.best_value:.4f}")

    final_hp = {
        **best_params,
        'max_bins': 255, 'early_stopping': True,
        'n_iter_no_change': 30, 'validation_fraction': 0.15, 'random_state': 42
    }

    # ── pick ONE production threshold from out-of-fold predictions,
    # rather than trusting a single fold's threshold ──
    tscv = TimeSeriesSplit(n_splits=5)
    oof_proba, oof_true = [], []

    for tr_idx, val_idx in tscv.split(X_tr_full):
        purged_tr_idx = tr_idx[:-horizon] if horizon > 0 else tr_idx
        if len(purged_tr_idx) < 50:
            continue
        y_tr_fold = y_bin_full[purged_tr_idx]
        if len(np.unique(y_tr_fold)) < 2:
            continue

        m = HistGradientBoostingClassifier(**final_hp)
        m.fit(X_tr_full.iloc[purged_tr_idx], y_tr_fold, sample_weight=w_clf_full[purged_tr_idx])
        oof_proba.append(m.predict_proba(X_tr_full.iloc[val_idx])[:, 1])
        oof_true.append(y_bin_full[val_idx])

    oof_proba = np.concatenate(oof_proba)
    oof_true = np.concatenate(oof_true)

    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, 33):
        f1 = f1_score(oof_true, (oof_proba >= t).astype(int), average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1

    clf_thresholds[col] = best_t
    print(f"  Chosen threshold: {best_t:.3f}  (OOF macro-F1={best_f1:.4f})")

    model = HistGradientBoostingClassifier(**final_hp)
    model.fit(X_tr_full, y_bin_full, sample_weight=w_clf_full)
    clf_models[col] = model
    print(f"  ✓ Complete!")

In [32]:
def combined_predict(col, X):
    proba = clf_models[col].predict_proba(X)[:, 1]
    sign = np.where(proba >= clf_thresholds[col], 1, -1)
    magnitude = np.abs(models[col].predict(X))
    return sign * magnitude

### iii. Evaluation

In [33]:
def evaluate_combined(label):
    rows = []
    for col in target_cols:
        y_true = y_test[col]
        valid = y_true.notna()
        y_true_valid = y_true[valid]
        y_bin_true = (y_true_valid > 0).astype(int)

        proba = clf_models[col].predict_proba(X_test[valid])[:, 1]
        clf_pred_bin = (proba >= clf_thresholds[col]).astype(int)

        y_pred_combined = combined_predict(col, X_test[valid])

        rows.append({
            "target": col,
            "auc": roc_auc_score(y_bin_true, proba),
            "f1": f1_score(y_bin_true, clf_pred_bin, average="macro", zero_division=0),
            "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred_combined)),
        })
    return pd.DataFrame(rows).assign(model=label)

combined_results = evaluate_combined("sign+mag")
print(combined_results.set_index("target").round(3))
print("\nMean:", combined_results[["auc","f1","rmse"]].mean().round(4).to_dict())

             auc     f1   rmse     model
target                                  
SPX_t+3    0.583  0.561  0.013  sign+mag
SPX_t+7    0.576  0.547  0.020  sign+mag
SPX_t+30   0.569  0.543  0.045  sign+mag
GOLD_t+3   0.499  0.506  0.016  sign+mag
GOLD_t+7   0.521  0.500  0.026  sign+mag
GOLD_t+30  0.512  0.478  0.042  sign+mag
VIX_t+3    0.554  0.520  0.091  sign+mag
VIX_t+7    0.612  0.565  0.138  sign+mag
VIX_t+30   0.622  0.571  0.242  sign+mag
TNX_t+3    0.486  0.473  0.020  sign+mag
TNX_t+7    0.488  0.487  0.036  sign+mag
TNX_t+30   0.397  0.453  0.073  sign+mag

Mean: {'auc': 0.5348, 'f1': 0.5169, 'rmse': 0.0635}


### iv. Significance Testing

In [34]:
spx_targets = ["SPX_t+3", "SPX_t+7", "SPX_t+30", "GOLD_t+3", "GOLD_t+7", "GOLD_t+30", "VIX_t+3", "VIX_t+7", "VIX_t+30"]

non_emb_cols = [c for c in X_train.columns if not c.startswith("emb_")]
X_train_noemb = X_train[non_emb_cols]
X_test_noemb  = X_test[non_emb_cols]

In [ ]:
models_noemb = {}
clf_models_noemb = {}
clf_thresholds_noemb = {}

for col in spx_targets:
    asset = col.split("_")[0]
    horizon = int(col.split("+")[1])
    y_col = y_train[col]
    valid = y_col.notna()

    X_tr_full = X_train_noemb[valid]
    y_tr_full = y_col[valid]
    y_bin_full = (y_tr_full > 0).astype(int).values
    w_tr_full = w_train[valid]

    # ── magnitude regressor (no-emb) ──
    study_r = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study_r.optimize(lambda trial: objective(trial, X_tr_full, y_tr_full, w_tr_full, asset, horizon),
                      n_trials=20, show_progress_bar=False)
    final_params = {**study_r.best_params, 'max_bins': 255, 'early_stopping': True,
                     'n_iter_no_change': 50, 'validation_fraction': 0.15, 'random_state': 42}
    m = HistGradientBoostingRegressor(**final_params)
    m.fit(X_tr_full, y_tr_full, sample_weight=w_tr_full)
    models_noemb[col] = m

    # ── sign classifier (no-emb) ──
    class_w = get_class_weight_multiplier(y_bin_full, neg_boost=1.0)
    w_clf_full = w_tr_full.values * class_w

    study_c = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study_c.optimize(lambda trial: objective_clf_purged(trial, X_tr_full, y_bin_full, w_clf_full, horizon),
                      n_trials=20, show_progress_bar=False)
    final_hp = {**study_c.best_params, 'max_bins': 255, 'early_stopping': True,
                'n_iter_no_change': 30, 'validation_fraction': 0.15, 'random_state': 42}

    # OOF threshold selection, same purge logic as before
    tscv = TimeSeriesSplit(n_splits=5)
    oof_proba, oof_true = [], []
    for tr_idx, val_idx in tscv.split(X_tr_full):
        purged_tr_idx = tr_idx[:-horizon] if horizon > 0 else tr_idx
        if len(purged_tr_idx) < 50 or len(np.unique(y_bin_full[purged_tr_idx])) < 2:
            continue
        mc = HistGradientBoostingClassifier(**final_hp)
        mc.fit(X_tr_full.iloc[purged_tr_idx], y_bin_full[purged_tr_idx], sample_weight=w_clf_full[purged_tr_idx])
        oof_proba.append(mc.predict_proba(X_tr_full.iloc[val_idx])[:, 1])
        oof_true.append(y_bin_full[val_idx])
    oof_proba, oof_true = np.concatenate(oof_proba), np.concatenate(oof_true)

    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, 33):
        f1 = f1_score(oof_true, (oof_proba >= t).astype(int), average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    clf_thresholds_noemb[col] = best_t

    mc_final = HistGradientBoostingClassifier(**final_hp)
    mc_final.fit(X_tr_full, y_bin_full, sample_weight=w_clf_full)
    clf_models_noemb[col] = mc_final
    print(f"✓ {col}  (threshold={best_t:.3f})")

In [36]:
def combined_predict_generic(col, X, clf_dict, thresh_dict, reg_dict):
    proba = clf_dict[col].predict_proba(X)[:, 1]
    sign = np.where(proba >= thresh_dict[col], 1, -1)
    magnitude = np.abs(reg_dict[col].predict(X))
    return sign * magnitude, proba

In [37]:
def eval_spx_stack(clf_dict, thresh_dict, reg_dict, X_te, label):
    rows = []
    for col in spx_targets:
        y_true = y_test[col]
        valid = y_true.notna()
        y_true_valid = y_true[valid]
        y_bin_true = (y_true_valid > 0).astype(int)

        y_pred, proba = combined_predict_generic(col, X_te[valid], clf_dict, thresh_dict, reg_dict)
        clf_pred_bin = (proba >= thresh_dict[col]).astype(int)

        rows.append({
            "target": col, "model": label,
            "auc": roc_auc_score(y_bin_true, proba),
            "f1": f1_score(y_bin_true, clf_pred_bin, average="macro", zero_division=0),
            "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred)),
        })
    return pd.DataFrame(rows)

spx_with_emb = eval_spx_stack(clf_models, clf_thresholds, models, X_test, "sign+mag_emb")
spx_no_emb   = eval_spx_stack(clf_models_noemb, clf_thresholds_noemb, models_noemb, X_test_noemb, "sign+mag_noemb")

comparison = pd.concat([spx_with_emb, spx_no_emb]).pivot(index="target", columns="model", values=["auc","f1","rmse"])
print(comparison.round(4))
print("\nMean")
print(pd.concat([spx_with_emb, spx_no_emb]).groupby("model")[["auc","f1","rmse"]].mean().round(4))

                   auc                          f1                 \
model     sign+mag_emb sign+mag_noemb sign+mag_emb sign+mag_noemb   
target                                                              
GOLD_t+3        0.4994         0.4835       0.5059         0.4365   
GOLD_t+30       0.5117         0.5409       0.4777         0.4861   
GOLD_t+7        0.5207         0.4462       0.4996         0.4464   
SPX_t+3         0.5828         0.6206       0.5607         0.5896   
SPX_t+30        0.5691         0.5131       0.5432         0.4846   
SPX_t+7         0.5758         0.5374       0.5467         0.5266   
VIX_t+3         0.5536         0.5483       0.5198         0.5210   
VIX_t+30        0.6216         0.6749       0.5710         0.5641   
VIX_t+7         0.6116         0.6326       0.5653         0.4968   

                  rmse                 
model     sign+mag_emb sign+mag_noemb  
target                                 
GOLD_t+3        0.0157         0.0158  
GOLD_t+30  

-> CI Testing EMB vs No EMB, Both separated predictor 

In [38]:
N_BOOT = 2000
rng = np.random.default_rng(42)
ci_rows = []

for col in spx_targets:
    y_true = y_test[col]
    valid = y_true.notna()
    y_true_valid = y_true[valid].reset_index(drop=True).values
    y_bin_true = (y_true_valid > 0).astype(int)
    n = len(y_true_valid)

    pred_with, proba_with = combined_predict_generic(col, X_test[valid], clf_models, clf_thresholds, models)
    pred_noemb, proba_noemb = combined_predict_generic(col, X_test_noemb[valid], clf_models_noemb, clf_thresholds_noemb, models_noemb)

    delta = {k: np.empty(N_BOOT) for k in ["auc", "f1", "rmse"]}
    b, attempts = 0, 0
    while b < N_BOOT and attempts < N_BOOT * 5:
        attempts += 1
        idx = rng.integers(0, n, size=n)
        y_b, y_b_bin = y_true_valid[idx], y_bin_true[idx]
        if len(np.unique(y_b_bin)) < 2:
            continue

        pw, pn = proba_with[idx], proba_noemb[idx]
        pwb = (pw >= clf_thresholds[col]).astype(int)
        pnb = (pn >= clf_thresholds_noemb[col]).astype(int)

        auc_w, auc_n = roc_auc_score(y_b_bin, pw), roc_auc_score(y_b_bin, pn)
        f1_w = f1_score(y_b_bin, pwb, average="macro", zero_division=0)
        f1_n = f1_score(y_b_bin, pnb, average="macro", zero_division=0)
        rmse_w = np.sqrt(mean_squared_error(y_b, pred_with[idx]))
        rmse_n = np.sqrt(mean_squared_error(y_b, pred_noemb[idx]))

        delta["auc"][b]  = auc_n - auc_w      # negative = embeddings help
        delta["f1"][b]   = f1_n - f1_w
        delta["rmse"][b] = rmse_n - rmse_w
        b += 1

    for k in delta: delta[k] = delta[k][:b]

    def ci(arr):
        lo, hi = np.percentile(arr, [5, 95])
        return f"[{lo:.4f}, {hi:.4f}]", (lo > 0 or hi < 0)

    row = {"target": col, "n_valid": n}
    for metric in ["auc", "f1", "rmse"]:
        ci_str, sig = ci(delta[metric])
        row[f"{metric}_delta_90ci"] = ci_str
        row[f"{metric}_significant"] = sig
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows).set_index("target")
print(ci_df)

           n_valid      auc_delta_90ci  auc_significant       f1_delta_90ci  \
target                                                                        
SPX_t+3        334   [-0.0317, 0.1067]            False   [-0.0301, 0.0913]   
SPX_t+7        334   [-0.0992, 0.0215]            False   [-0.0749, 0.0344]   
SPX_t+30       334  [-0.1011, -0.0124]             True  [-0.1078, -0.0099]   
GOLD_t+3       334   [-0.0840, 0.0499]            False  [-0.1335, -0.0097]   
GOLD_t+7       334   [-0.1480, 0.0009]            False   [-0.1142, 0.0053]   
GOLD_t+30      334   [-0.0345, 0.0950]            False   [-0.0483, 0.0652]   
VIX_t+3        334   [-0.0716, 0.0609]            False   [-0.0635, 0.0607]   
VIX_t+7        334   [-0.0374, 0.0777]            False  [-0.1240, -0.0104]   
VIX_t+30       334    [0.0083, 0.0963]             True   [-0.0543, 0.0401]   

           f1_significant     rmse_delta_90ci  rmse_significant  
target                                                          

# 5. Save Model

In [39]:
import json
from datetime import datetime, timezone

PROD_DIR = "models_el/production"
os.makedirs(f"{PROD_DIR}/magnitude", exist_ok=True)
os.makedirs(f"{PROD_DIR}/sign", exist_ok=True)

for col, model in models.items():
    joblib.dump(model, f"{PROD_DIR}/magnitude/{col}.pkl")

for col, model in clf_models.items():
    joblib.dump(model, f"{PROD_DIR}/sign/{col}.pkl")

with open(f"{PROD_DIR}/thresholds.json", "w") as f:
    json.dump(clf_thresholds, f, indent=2)

joblib.dump(list(X_train.columns), f"{PROD_DIR}/feature_columns.pkl")

with open(f"{PROD_DIR}/metadata.json", "w") as f:
    json.dump({
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "split_date": split_date,
        "train_start": str(train_df["date"].min().date()),
        "train_end": str(train_df["date"].max().date()),
        "test_start": str(test_df["date"].min().date()),
        "test_end": str(test_df["date"].max().date()),
        "targets": target_cols,
    }, f, indent=2)

print(f"✓ Saved {len(models)} magnitude models, {len(clf_models)} sign models, thresholds, feature columns, and metadata")

✓ Saved 12 magnitude models, 12 sign models, thresholds, feature columns, and metadata
